# Example

This notebook demonstrates how to use Wags-LLM using the `BedrockClaudeJsonClient`.

In [1]:
import logging
import sys

from pydantic import BaseModel

from wags_llm.cache import InMemoryCache
from wags_llm.client.bedrock import BedrockClaudeJsonClient
from wags_llm.prompts.base import BasePromptTemplate
from wags_llm.prompts.registry import PromptRegistry
from wags_llm.services.json_task import JsonTaskService

logging.basicConfig(
    stream=sys.stdout,
    level=logging.WARNING,
    format="%(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("wags_llm").setLevel(logging.DEBUG)

Let's pretend we want to find the MONDO identifier for a given free-text label. We want a simple response to be `mondo_id` as a string.

In [ ]:
class MyPrompt(BasePromptTemplate):
    name = "mondo_id_classification"
    version = "v1"

    def build_system_prompt(self):
        return "Return ONLY a valid JSON object. Do not use markdown code fences. {'mondo_id': str}"

    def build_user_prompt(self, payload):
        return f"Input:\n{payload['text']}"


class Result(BaseModel):
    mondo_id: str

We will be using Claude Sonnet 4.6 and will demonstrate how to use a cache (optional).

In [3]:
client = BedrockClaudeJsonClient(
    model_id="us.anthropic.claude-sonnet-4-6",
    region_name="us-east-1",
    profile_name="dev-account",
)

registry = PromptRegistry()
registry.register(MyPrompt())

service = JsonTaskService(
    client=client,
    prompt_registry=registry,
    cache=InMemoryCache(),
)

wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=300, temperature=0.000000
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.prompts.registry - DEBUG - Registering prompt: name='mondo_id_classification', version='v1'


In [4]:
result = service.run(
    prompt_name="mondo_id_classification",
    prompt_version="v1",
    payload={"text": "melanoma"},
    response_model=Result,
)
result

wags_llm.services.json_task - DEBUG - Cache lookup using key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5' (for cache_payload={'payload': {'text': 'melanoma'}, 'model': 'us.anthropic.claude-sonnet-4-6', 'prompt_name': 'mondo_id_classification', 'prompt_version': 'v1'})
wags_llm.cache.in_memory - DEBUG - Cache miss for cache key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5'
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 36, 'outputTokens': 16, 'totalTokens': 52, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 1440}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'text': '{"mondo_id": "MONDO:0005105"}'}]


Result(mondo_id='MONDO:0005105')

We can see that the cache works when running the same cell again

In [5]:
result = service.run(
    prompt_name="mondo_id_classification",
    prompt_version="v1",
    payload={"text": "melanoma"},
    response_model=Result,
)
result

wags_llm.services.json_task - DEBUG - Cache lookup using key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5' (for cache_payload={'payload': {'text': 'melanoma'}, 'model': 'us.anthropic.claude-sonnet-4-6', 'prompt_name': 'mondo_id_classification', 'prompt_version': 'v1'})
wags_llm.cache.in_memory - DEBUG - Cache hit for cache key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5'


Result(mondo_id='MONDO:0005105')